In [ ]:
import pandas as pd
import numpy as np
import boto3
from sklearn.model_selection import train_test_split
import sagemaker
from sagemaker import Session
import io
from sagemaker.amazon.common import smac
import os
from sagemaker.amazon.amazon_estimator import get_image_uri

In [ ]:
df = pd.read_csv("student_scores.csv")
df.head()

In [ ]:
df.shape

In [ ]:
# separate x and y
x=df[["Hours"]]
y=df[["scores"]]

In [ ]:
# checking data types
x.dtypes
x = x.astype("float32")
y = y.astype("float32")

In [ ]:
y.dtypes

In [ ]:
# split the data
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size=0.2)

In [ ]:
# reset index
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [ ]:
X_train

In [ ]:
#we need to take label column as vector
y_train = y_train.iloc[:,0]

In [ ]:
y_train

In [ ]:
y_test = y_test.iloc[:,0]

* 建立 AWS s3 bucket

<img src="../figures/sagemaker/aws/aws_purposebuckets.png" width=100%><br>
- 進入 Buckets 頁面，點選 Create bucket

<img src="../figures/sagemaker/aws/aws_createbuckett.png" width=85%><br>
- Bucket name：bappy-bucket-25

<img src="../figures/sagemaker/aws/aws_createbucketb.png" width=85%><br>
- 頁面下拉，然後建立

<img src="../figures/sagemaker/aws/aws_purposebuckets2.png" width=85%><br>
- 回到 Buckets 頁面，點選剛建立 Bucket。

In [ ]:
#Lets create sagemaker session
sagemaker_session = sagemaker.Session()
#define the bucket name
bucket_name = "bappy-sagemaker"
#define the prefix
prefix = "linear-learner"
#get the execution role
role = sagemaker.get_execution_role()

In [ ]:
#convert X_train into numpy array
X_train = np.array(X_train)

In [ ]:
#create the buffer 

buf=io.BytesIO() # 建立一個記憶體緩衝區

# buf: 目標檔案對象（通常是 BytesIO 或實體檔案）
# X_train: 你的 NumPy 陣列（特徵矩陣）
# y_train: 對應的標籤 NumPy 陣列（選填）
smac.write_numpy_to_dense_tensor(buf,X_train,y_train) # 將 NumPy 數據寫入緩衝區並轉換為 Dense Tensor 格式

buf.seek(0) # 指標回到緩衝區開頭，準備讀取上傳

In [ ]:
#define the name of the file
key="student-data"

#code to upload in s3
boto3.resource('s3').Bucket(bucket_name).Object(os.path.join(prefix,'train',key)).upload_fileobj(buf)

#path of our data
s3_train_data=f"s3://{bucket_name}/{prefix}/train/{key}"

print("Data uploaded",s3_train_data)

<img src="../figures/sagemaker/aws/aws_bucketdashboard.png" width=90%><br>
- 到 AWS bucket 頁面，點選物件 linear-learner

<img src="../figures/sagemaker/aws/aws_bucketllobject.png" width=90%><br>
- 點選 train 資料夾

<img src="../figures/sagemaker/aws/aws_buckettraindir.png" width=90%><br>
- 檢視 train 資料夾

In [ ]:
#convert X_test to numpy array
X_test = np.array(X_test)

#create the buffer 

buf=io.BytesIO() # 建立一個記憶體緩衝區

smac.write_numpy_to_dense_tensor(buf,X_test,y_test) # 將 NumPy 數據寫入緩衝區並轉換為 Dense Tensor 格式

buf.seek(0) # 指標回到緩衝區開頭，準備讀取上傳

#define the name of the file
key="student-data-test"

#code to upload in s3
boto3.resource('s3').Bucket(bucket_name).Object(os.path.join(prefix,'test',key)).upload_fileobj(buf)

#path of our data
s3_test_data=f"s3://{bucket_name}/{prefix}/test/{key}"

print("Data uploaded",s3_test_data)

<img src="../figures/sagemaker/aws/aws_bucketllobject2.png" width=90%><br>
- 點選 test 資料夾

<img src="../figures/sagemaker/aws/aws_buckettestdir.png" width=90%><br>
- 檢視 test 資料夾

In [ ]:
#output location
output_location = f"s3://{bucket_name}/{prefix}/output"
output_location

In [ ]:
#bring the container
container=sagemaker.image_uris.retrieve("linear-learner",boto3.Session().region_name)

In [ ]:
#define the estimator
linear=sagemaker.estimator.Estimator(container,role,instance_count=1,instance_type='ml.c4.xlarge',output_path=output_location,sagemaker_session=sagemaker_session)

In [ ]:
#setting up the hyper parameters
linear.set_hyperparameters(feature_dim=1,predictor_type="regressor",mini_batch_size=4,epochs=6,num_models=32,loss="absolute_loss")

In [ ]:
#fit the model (此步驟需花多些時間)
linear.fit({"train":s3_train_data})

In [ ]:
#deploy the model
linear_regresor=linear.deploy(initial_instance_count=1,instance_type="ml.m4.xlarge")

<img src="../figures/sagemaker/aws/aws_sagemakerinstance.png" width=95%><br>
- 在 SageMaker AI 的 dashboard 中，inference 的 Endpoints 下，可以檢視部署完成的 endpoint。
- 越新的，列在越下方。

In [ ]:
linear_regresor.serializer=sagemaker.serializers.CSVSerializer()
linear_regresor.deserializer=sagemaker.deserializers.JSONDeserializer()

In [ ]:
#prediction
results=linear_regresor.predict(X_test)

In [ ]:
results

results 的樣態如下：
<pre>
{
    'predictions':[{'score':37.19...},
        {'score':24.09...},
        {'score':63.41...},
        ...
    ]
}
</pre>

In [ ]:
predictions=np.array([i["score"] for i in results["predictions"]])

In [ ]:
predictions

In [ ]:
linear_regresor.delete_endpoint()

<img src="../figures/sagemaker/aws/aws_sagemakerinstance2.png" width=100%><br>
- 在 SageMaker AI 的 dashboard 中，inference 的 Endpoints 下，可以檢視之前部署的 endpoint 已經被刪除了。

* 若要刪除 Sagemaker 中的 instance

    <img src="../figures/sagemaker/aws/aws_instancestopspace.png" width=100%><br>
    - 在 Sagemaker dashboard 中點選 Stop space

    <img src="../figures/sagemaker/aws/aws_instancestopspacec.png" width=75%><br>
    - 按下 Stop space 以確認

    <img src="../figures/sagemaker/aws/aws_instancestopspaced.png" width=100%><br>
    - 再回到 Sagemaker dashboard 中，點選 ⋮ 後，點選 Delete space

    <img src="../figures/sagemaker/aws/aws_jupyterlabspace.png" width=100%><br>
    - 刪除 space 後的的頁面

* 刪除 bucket

  <img src="../figures/sagemaker/aws/aws_bucketdelete.png" width=100%><br>
  - 點選欲刪除的 bucket，點選 delete
  - 注意，bucket 內必須是空的，若有物件存在，必須先刪除，否則 bucket 無法刪除